In [1]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display, Markdown

from telegram_scraper import study, viz
from telegram_scraper.study import CHANNELS, CHANNEL_ORDER, SHORT_NAMES, WINDOW
from telegram_scraper.analysis import stats

viz.apply_style()
np.random.seed(study.SEED)
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)
posts = study.load_posts()
events = study.verified_events(posts)
f"{len(posts):,} posts from {posts['channel_slug'].nunique()} channels, {posts['timestamp'].min():%Y-%m-%d} to {posts['timestamp'].max():%Y-%m-%d}"

'12,221 posts from 6 channels, 2026-02-28 to 2026-04-17'

# 00 · Collect: where the data comes from

> **Question.** What exactly is in the Telegram snapshot every other notebook reads, and can anyone reproduce it?
>
> **Answer.** One frozen snapshot of 14,620 Telegram messages from six channels (12,221 posts once albums are grouped, 28 Feb - 17 Apr 2026, 49 UTC days), fetched once and never re-fetched by the notebooks.

## 1. The study design: a 2 × 2 of ownership × country

Six public Telegram channels, chosen so that state vs. private ownership can be compared *within* each country. Two caveats are part of the design, not footnotes: the IDF channel is state-owned but is a military spokesperson, not a broadcaster, and ownership of Jewish Breaking News (JBN) could not be verified, so it is labelled "private (aggregator)".

In [2]:
design = study.channel_table()[["channel_slug", "name", "country", "ownership_label", "role"]]
design

,channel_slug,name,country,ownership_label,role
0,presstv,Press TV,Iran,state (broadcaster),IRIB English-language state broadcaster
1,irna-en,IRNA (English),Iran,state (news agency),"Iranian state news agency, English service"
2,idfofficial,Israel Defense Forces,Israel,state (military),"Military spokesperson channel (state, but not a broadcaster)"
3,iltvnews,ILTV Israel News,Israel,private (TV/online),Private English-language TV and online news
4,thetimesofisrael2022,The Times of Israel,Israel,private (newspaper),Private newspaper; photo posts with a headline and a 'Read more' link
5,jewishbreakingnewstelegram,Jewish Breaking News,Israel,private (aggregator),Israel-focused news aggregator; ownership unverified


## 2. The snapshot command

The data were fetched once, live, with the command below and then frozen on disk. Every notebook loads the frozen parquet files; none of them talks to Telegram. The fetch cell here is guarded by `RUN_FETCH = False` so that re-executing the notebook can never silently overwrite or add to the evidence.

In [3]:
RUN_FETCH = False  # keep False: the study reads the frozen snapshot below

SNAPSHOT_COMMAND = (
    "uv run telegram-scraper snapshot "
    + " ".join(f"--channel {slug}" for slug in CHANNEL_ORDER)
    + " --start 2026-02-28 --end 2026-04-18 --out data/raw"
)
print(SNAPSHOT_COMMAND)

if RUN_FETCH:
    import subprocess
    subprocess.run(SNAPSHOT_COMMAND.split(), cwd=study.ROOT, check=True)
else:
    print("RUN_FETCH is False -> using", study.SNAPSHOT_DIR.relative_to(study.ROOT))

uv run telegram-scraper snapshot --channel presstv --channel irna-en --channel idfofficial --channel iltvnews --channel thetimesofisrael2022 --channel jewishbreakingnewstelegram --start 2026-02-28 --end 2026-04-18 --out data/raw
RUN_FETCH is False -> using data/raw/2026-02-28_2026-04-18_20261002T081854Z


## 3. Manifest of the frozen snapshot

In [4]:
import json
manifest = json.loads((study.SNAPSHOT_DIR / "manifest.json").read_text())
print({k: v for k, v in manifest.items() if k != "channels"})
manifest_df = pd.DataFrame(manifest["channels"]).drop(columns=["chat_id", "min_id", "max_id"])
manifest_df

{'run_id': '2026-02-28_2026-04-18_20261002T081854Z', 'fetched_at': '2026-10-02T08:18:55.837498+00:00', 'start': '2026-02-28T00:00:00+00:00', 'end': '2026-04-18T00:00:00+00:00', 'download_media': False, 'git_sha': 'ce2bf45bb3a03efc9470028f696af61b5fd26ba9', 'package_version': '0.1.0'}


,slug,title,username,message_count,first_ts,last_ts
0,presstv,Press TV,presstv,8574,2026-02-28T00:01:46+00:00,2026-04-17T23:28:28+00:00
1,irna-en,IRNA News Agency 🇮🇷,Irna_en,2010,2026-02-28T05:10:43+00:00,2026-04-17T17:20:29+00:00
2,idfofficial,Israel Defense Forces,idfofficial,1429,2026-02-28T04:37:42+00:00,2026-04-17T16:46:14+00:00
3,iltvnews,ILTV Israel News 24/7,ILTVnews,691,2026-02-28T07:35:40+00:00,2026-04-17T12:28:12+00:00
4,thetimesofisrael2022,The Times of Israel,TheTimesOfIsrael2022,622,2026-02-28T06:24:43+00:00,2026-04-17T19:25:16+00:00
5,jewishbreakingnewstelegram,Jewish Breaking News - JBN,JewishBreakingNewsTelegram,1294,2026-03-01T00:28:39+00:00,2026-04-17T14:27:02+00:00


## 4. Messages vs. posts per channel

Telegram stores each item of an album as its own message. The study counts a **post** = one message or one album, so a ten-photo album is one post. Counts below are the denominators used everywhere else.

In [5]:
messages = study.load_snapshot(study.SNAPSHOT_DIR)
counts = (
    messages.groupby("channel_slug").size().rename("messages").to_frame()
    .join(posts.groupby("channel_slug").size().rename("posts"))
    .join(posts.groupby("channel_slug")["has_clean_text"].sum().rename("posts_with_text"))
    .reindex(CHANNEL_ORDER)
)
counts["items_per_post"] = counts["messages"] / counts["posts"]
counts.insert(0, "channel", [SHORT_NAMES[s] for s in counts.index])
display(counts)
print("total messages:", int(counts["messages"].sum()), "| total posts:", int(counts["posts"].sum()))
assert counts["posts"].sum() == len(posts)
assert posts["timestamp"].between(*study.WINDOW, inclusive="left").all()

,channel,messages,posts,posts_with_text,items_per_post
channel_slug,,,,,
presstv,Press TV,8574,6683,6619,1.282957
irna-en,IRNA,2010,1637,1635,1.227856
idfofficial,IDF,1429,1429,1425,1.000000
iltvnews,ILTV,691,556,542,1.242806
thetimesofisrael2022,ToI,622,622,622,1.000000
jewishbreakingnewstelegram,JBN,1294,1294,1113,1.000000


total messages: 14620 | total posts: 12221


In [6]:
_ = study.record_findings("00_collect", {
    "snapshot_run_id": manifest["run_id"],
    "fetched_at": manifest["fetched_at"],
    "window": ["2026-02-28", "2026-04-18 (exclusive)"],
    "n_messages": int(counts["messages"].sum()),
    "n_posts": int(counts["posts"].sum()),
    "per_channel": counts.drop(columns="channel").reset_index().to_dict("records"),
    "snapshot_command": SNAPSHOT_COMMAND,
})
_ = viz.export_data(counts.reset_index(), "00_collect_counts")

## What this means for the article

- **Cite the denominators from this table**: Press TV 6,683 posts (55% of the corpus), IRNA 1,637, IDF 1,429, JBN 1,294, ToI 622, ILTV 556. Press TV alone is more than the other five channels combined.
- **Say "posts", not "messages".** The raw message counts (Press TV 8,574; IRNA 2,010; ILTV 691) include one row per album item and overstate three channels by 23-28%.
- **Coverage caveat.** JBN has no posts on 28 Feb (its first post is 1 Mar 00:28 UTC), so the war's first day is missing for that channel; the snapshot cannot tell us whether the channel was silent or posts were later deleted. A handful of other zero or near-zero days are listed in notebook 02.
- **Reproducibility.** The snapshot run id (`2026-02-28_2026-04-18_20261002T081854Z`) and its git SHA are recorded in `findings.json`; the fetch was done once, on 2 Oct 2026, months after the events. Views and forward counts are therefore *late* counts, not what readers saw in April.